# The data: a table and a map

```{admonition} Question
:class: tip
What does one row of our table look like on the map?
```

**In this notebook:** 1 · load the table  2 · read five rows  3 · put them on the map  4 · see where the label comes from  5 · compare neighbours

## 1 · Load the table

The Helsinki region as 1,459 hexagons of the H3 grid (week 7), each about 0.57 km². One hexagon is one **row**. Everything comes from OpenStreetMap.

In [ ]:
import numpy as np
import pandas as pd

import helpers as h

hexes, X, y, W = h.load()        # the table, the features, the labels, the neighbours
print(f"{len(hexes):,} hexagons, {X.shape[1]} features")
print(pd.Series(y).value_counts().to_string())

- `X` · the **features**: 16 counts of points of interest (POIs) per hexagon, as `log(1 + count)`.
- `y` · the **label**: the land use, one of `green`, `residential`, `work`.
- `W` · which hexagons touch (the spatial weights matrix of week 3).

## 2 · Read five rows

Five hexagons that every notebook follows, and three of the 16 features.

In [ ]:
five = h.five(hexes)             # place name -> row number
rows = hexes.loc[five.values, ["car", "play", "office", "poi_total", "landuse"]]
rows.index = five.index
rows

`car` counts parking and fuel, `play` playgrounds and picnic sites, `office` offices. The counts are what we know about a place. The last column is what a model should predict.

All 16 counts of one row, the hexagon of the Otaniemi campus:

In [ ]:
h.profile(hexes, five["Otaniemi"]);

## 3 · The same rows on the map

A table has no geography: shuffle its rows and nothing changes. On the map every row has a place.

In [ ]:
ax = h.map_classes(hexes, y, title="The land-use label, and the five rows", places=False)
h.number_hexagons(ax, hexes, five.values)

Point at a hexagon to read its row. Zoom in to see the streets and parks behind the counts.

<iframe src="_static/maps/hexagons.html" class="webmap" title="Interactive map of the hexagons"></iframe>

*Map data and background map © OpenStreetMap contributors.*

## 4 · The label is a rule

A hexagon holds several land uses and gets one name: at least 20% built-up land makes it `residential` or `work`, whichever covers more. Everything else is `green`. What if the rule were 10% or 30%?

In [ ]:
for rule in (0.10, 0.30):
    changed = np.mean(h.label_with(hexes, rule) != y)
    print(f"{rule:.0%} rule: {changed:.0%} of the labels change")

**Result.** Another reasonable rule renames about one hexagon in eight. The label is a decision, not a measurement, so no model should reach 100%.

## 5 · Neighbours are alike

How often does a hexagon have the same label as a hexagon that touches it?

In [ ]:
same = np.mean([np.mean(y[W[i].indices] == y[i]) for i in range(len(y)) if W[i].nnz])
chance = (pd.Series(y).value_counts(normalize=True) ** 2).sum()
print(f"same label as a neighbour: {same:.0%}     if labels were scattered at random: {chance:.0%}")

**Result.** 69% against 42%: spatial dependence in one number (Tobler's first law, week 3). A hexagon without a label is not a hexagon about which we know nothing.

**Next:** [what learning is](learning), or straight to [supervised learning](supervised).